# Synthetic Datasets: Plots of Utility and Group Fairness Metrics

Author: Ilse Harmers \
Last modified: February 21, 2026

In [ ]:
# Importing libraries.
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
from matplotlib import ticker
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import matplotlib
params = {'axes.titlesize':'18',
          'xtick.labelsize':'15',
          'ytick.labelsize':'15',
          'font.size':'16',
          'legend.fontsize':'medium',
          'lines.linewidth':'2.0',
          'font.weight':'normal',
          'lines.markersize':'5',
          'text.latex.preamble': r'\usepackage{amsfonts}',
          }
matplotlib.rcParams.update(params)
plt.rcParams["mathtext.fontset"] = "cm"
plt.rc('text', usetex=True)
plt.rc('font', family='serif')

## Initialization

In [ ]:
# Initializing the values for epsilon, the labels for the different metrics and our functions. 
epsi = [1, 2, 5, 8]
metrics = ["acc", "prec", "recall", "auroc", "dem", "dis", "eqopp"]

def results_analysis(file_path, epsilons):
    """This function returns the 'best' synthetic dataset out of all our runs for each epsilon value in 'epsilons'. 'Best' refers to the dataset with 
    the highest averaged AUROC score. This function also returns the group fairness metrics and accuracy scores of this synthetic dataset, and the 
    averaged results of the GAN model across all the runs for each epsilon value separately.
    
    file_path [string]: folder location of the GAN's synthetic datasets
    epsilons [list]: epsilon values during GAN training
    """
    
    results_eps_avg = {}
    results_eps_avg_std = {}
    results_eps_best = {}
    results_eps_best_std = {}
    eps_bests = []
    
    for e in epsilons:
        # Importing synthetic datasets' results as pandas DataFrames.
        run1 = pd.read_csv(file_path + f"epsi_{e}/run1/results_epsi-{e}_run1.csv", index_col = 0, na_values = "np.nan")
        run2 = pd.read_csv(file_path + f"epsi_{e}/run2/results_epsi-{e}_run2.csv", index_col = 0, na_values = "np.nan")
        run3 = pd.read_csv(file_path + f"epsi_{e}/run3/results_epsi-{e}_run3.csv", index_col = 0, na_values = "np.nan")
        run4 = pd.read_csv(file_path + f"epsi_{e}/run4/results_epsi-{e}_run4.csv", index_col = 0, na_values = "np.nan")
        run5 = pd.read_csv(file_path + f"epsi_{e}/run5/results_epsi-{e}_run5.csv", index_col = 0, na_values = "np.nan")

        # Concatenating the results into one DataFrame. 
        results = pd.concat([run1.reset_index(drop = True), run2.reset_index(drop = True),
                             run3.reset_index(drop = True), run4.reset_index(drop = True),
                             run5.reset_index(drop = True)], axis = 0)
        
        eps_avg = {}
        eps_avg_std = {}
        
        eps_avg["dem-parity"] = np.nanmean(abs(results[["dem-parity"]]), axis = 1)
        eps_avg["dis-impact"] = np.nanmean(results[["dis-impact"]], axis = 1)
        eps_avg_std["dem-parity"] = np.nanstd(abs(results[["dem-parity"]]), axis = 1)#results["dem-parity"].std(ddof = 0)
        eps_avg_std["dis-impact"] = np.nanstd(results[["dis-impact"]], axis = 1)#results["dis-impact"].std(ddof = 0)
    
        for m in metrics:
            if m == "dis":   # averaging over 'regular' values
                m_mean = np.nanmean(results[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]], axis = 1)
                m_std = np.nanstd(results[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]], axis = 1)
            else:   # averaging over absolute values
                m_mean = np.nanmean(abs(results[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]), axis = 1) 
                m_std = np.nanstd(abs(results[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]), axis = 1)
            eps_avg[m] = m_mean
            eps_avg_std[m] = m_std
    
        results_eps_avg[f"epsi_{e}"] = eps_avg
        results_eps_avg_std[f"epsi_{e}"] = eps_avg_std
        
        max_auroc_idx = results[["rf-auroc", "mlp-auroc", "lda-auroc"]].mean(axis = 1).argmax()
        print(f"Synthetic dataset with highest AUROC in epsi_{e}: sample {max_auroc_idx + 1}")
    
        eps_best = results.iloc[max_auroc_idx].copy()
        eps_bests.append(eps_best)
    
        eps_best_avg = {}
        eps_best_avg_std = {}
        
        eps_best_avg["dem-parity"] = abs(eps_best["dem-parity"])
        eps_best_avg["dis-impact"] = eps_best["dis-impact"]
        eps_best_avg_std["dem-parity"] = "NaN"
        eps_best_avg_std["dis-impact"] = "NaN"
    
        for m in metrics:
            if m == "dis":   # averaging over 'regular' values
                m_mean = np.nanmean(eps_best[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]])
                m_std = np.nanstd(eps_best[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]])
            else:   # averaging over absolute values
                m_mean = np.nanmean(abs(eps_best[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]])) 
                m_std = np.nanstd(abs(eps_best[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]))
            eps_best_avg[m] = m_mean
            eps_best_avg_std[m] = m_std
        
        results_eps_best[f"epsi_{e}"] = eps_best_avg 
        results_eps_best_std[f"epsi_{e}"] = eps_best_avg_std

    return (eps_bests, pd.DataFrame(results_eps_best).T, pd.DataFrame(results_eps_best_std).T, 
            pd.DataFrame(results_eps_avg).T, pd.DataFrame(results_eps_avg_std).T)

def metrics_average(adult_samp, bank_samp, credit_samp, epsi = epsi):
    """This function computes the average classifier utility and group fairness results of the Adult, Bank and Credit 'best' synthetic datasets for
    each epsilon value. Both the means and standard deviations of the averaged results are returned."""

    avg_met_results = {}
    avg_met_results_std = {}
    for e in range(len(epsi)):
        eps_avg = {}
        eps_avg_std = {}
        for m in metrics:
            if m != "dis":   # averaging over absolute values
                avg = [abs(adult_samp.iloc[e][[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]), abs(bank_samp.iloc[e][[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]),
                       abs(credit_samp.iloc[e][[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]])]
            else:   # averaging over 'regular' values
                avg = [adult_samp.iloc[e][[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]], bank_samp.iloc[e][[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]],
                       credit_samp.iloc[e][[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]]
            eps_avg[m] = np.nanmean(avg)
            eps_avg_std[m] = np.nanstd(avg)
        avg_met_results[f"epsi_{epsi[e]}"] = eps_avg 
        avg_met_results_std[f"epsi_{epsi[e]}"] = eps_avg_std

    avg_met_results = pd.DataFrame(avg_met_results).T
    avg_met_results_std = pd.DataFrame(avg_met_results_std).T

    return avg_met_results, avg_met_results_std

## Adult Dataset (Real)

In [ ]:
adult_results_real = pd.read_csv("./train-test-datasets/adult/results_real.csv", index_col = 0)

results_real_avg = {}
for m in metrics:
    if m == "dis":   # averaging over 'regular' values
        m_mean = adult_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]].mean().mean()
        m_std = adult_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]].mean().std(ddof = 0)
    else:   # averaging over absolute values
        m_mean = abs(adult_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]).mean().mean()
        m_std = abs(adult_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]).mean().std(ddof = 0)
    results_real_avg[m] = (m_mean, m_std)

adult_results = pd.concat([adult_results_real[["dem-parity", "dis-impact"]].reset_index(drop = True),
                           pd.DataFrame(results_real_avg).reset_index(drop = True)], 
                           axis = 1).rename(index = {0: "mean", 1: "std"})

In [ ]:
adult_results

## Synthetic Adult Dataset (PATE-GAN)

In [ ]:
# Assigning file path for importing the datasets. 
file_path = "synthetic-datasets_PATE-GAN/adult/"
adult_eps_bests_PT, adult_results_best_PT, adult_results_best_PT_std, results_eps_avg, results_eps_avg_std = results_analysis(file_path, 
                                                                                                                              epsilons = epsi)

In [ ]:
adult_results_best_PT

## Synthetic Adult Dataset (DP-GAN)

In [ ]:
# Assigning file path for importing the datasets. 
file_path = "synthetic-datasets_DP-GAN_B=512/adult/"
adult_eps_bests_DP, adult_results_best_DP, adult_results_best_DP_std, results_eps_avg, results_eps_avg_std = results_analysis(file_path,
                                                                                                                              epsilons = epsi)

In [ ]:
adult_results_best_DP

## Synthetic Adult Dataset (WGAN-GP)

In [ ]:
# Assigning file path for importing the datasets. 
file_path = "synthetic-datasets_WGAN-GP/adult/"

# Importing synthetic datasets' results as pandas DataFrames.
run1 = pd.read_csv(file_path + f"run1/results_run1.csv", index_col = 0, na_values = "np.nan")
run2 = pd.read_csv(file_path + f"run2/results_run2.csv", index_col = 0, na_values = "np.nan")
run3 = pd.read_csv(file_path + f"run3/results_run3.csv", index_col = 0, na_values = "np.nan")
run4 = pd.read_csv(file_path + f"run4/results_run4.csv", index_col = 0, na_values = "np.nan")
run5 = pd.read_csv(file_path + f"run5/results_run5.csv", index_col = 0, na_values = "np.nan")

# Concatenating the results into one DataFrame. 
results = pd.concat([run1.reset_index(drop = True), run2.reset_index(drop = True),
                     run3.reset_index(drop = True), run4.reset_index(drop = True),
                     run5.reset_index(drop = True)], axis = 0)
    
avg = {}
avg["dem-parity"] = (results["dem-parity"].mean(), results["dem-parity"].std())
avg["dis-impact"] = (results["dis-impact"].mean(), results["dis-impact"].std())
    
for m in metrics:
    if m == "dis":   # averaging over 'regular' values
        m_mean = np.nanmean(results[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]], axis = 1)
        m_std = np.nanstd(results[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]], axis = 1)
    else:   # averaging over absolute values
        m_mean = np.nanmean(abs(results[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]), axis = 1) 
        m_std = np.nanstd(abs(results[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]), axis = 1)
    avg[m] = (m_mean, m_std)
    
max_auroc_idx = results[["rf-auroc", "mlp-auroc", "lda-auroc"]].mean(axis = 1).argmax()
print(f"Synthetic dataset with highest AUROC: sample {max_auroc_idx + 1}")
    
adult_wgangp_set = pd.DataFrame(results.iloc[max_auroc_idx].copy()).T
    
best_avg = {}
    
for m in metrics:
    if m == "dis":   # averaging over 'regular' values
        m_mean = np.nanmean(adult_wgangp_set[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]])
        m_std = np.nanstd(adult_wgangp_set[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]])
    else:   # averaging over absolute values
        m_mean = np.nanmean(abs(adult_wgangp_set[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]])) 
        m_std = np.nanstd(abs(adult_wgangp_set[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]))
    best_avg[m] = (m_mean, m_std)

adult_wgangp_results = pd.concat([adult_wgangp_set[["dem-parity", "dis-impact"]].reset_index(drop = True),
                                  pd.DataFrame(best_avg).reset_index(drop = True)], 
                                  axis = 1).rename(index = {0: "mean", 1: "std"})

In [ ]:
adult_wgangp_results

## Bank Marketing Dataset (Real)

In [ ]:
bank_results_real = pd.read_csv("./train-test-datasets/bank-marketing/results_real.csv", index_col = 0)

results_real_avg = {}
for m in metrics:
    if m == "dis":   # averaging over 'regular' values
        m_mean = bank_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]].mean().mean()
        m_std = bank_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]].mean().std(ddof = 0)
    else:   # averaging over absolute values
        m_mean = abs(bank_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]).mean().mean()
        m_std = abs(bank_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]).mean().std(ddof = 0)
    results_real_avg[m] = (m_mean, m_std)

bank_results = pd.concat([bank_results_real[["dem-parity", "dis-impact"]].reset_index(drop = True),
                          pd.DataFrame(results_real_avg).reset_index(drop = True)], 
                          axis = 1).rename(index = {0: "mean", 1: "std"})

In [ ]:
bank_results

## Synthetic Bank Marketing Dataset (PATE-GAN)

In [ ]:
# Assigning file path for importing the datasets. 
file_path = "synthetic-datasets_PATE-GAN/bank-marketing/"
bank_eps_bests_PT, bank_results_best_PT, bank_results_best_PT_std, results_eps_avg, results_eps_avg_std = results_analysis(file_path, epsilons = epsi)

In [ ]:
bank_results_best_PT

## Synthetic Bank Marketing Dataset (DP-GAN)

In [ ]:
# Assigning file path for importing the datasets. 
file_path = "synthetic-datasets_DP-GAN_B=512/bank-marketing/"
bank_eps_bests_DP, bank_results_best_DP, bank_results_best_DP_std, results_eps_avg, results_eps_avg_std = results_analysis(file_path, epsilons = epsi)

In [ ]:
bank_results_best_DP

## Synthetic Bank Marketing Dataset (WGAN-GP)

In [ ]:
# Assigning file path for importing the datasets. 
file_path = "synthetic-datasets_WGAN-GP/bank-marketing/"

# Importing synthetic datasets' results as pandas DataFrames.
run1 = pd.read_csv(file_path + f"run1/results_run1.csv", index_col = 0, na_values = "np.nan")
run2 = pd.read_csv(file_path + f"run2/results_run2.csv", index_col = 0, na_values = "np.nan")
run3 = pd.read_csv(file_path + f"run3/results_run3.csv", index_col = 0, na_values = "np.nan")
run4 = pd.read_csv(file_path + f"run4/results_run4.csv", index_col = 0, na_values = "np.nan")
run5 = pd.read_csv(file_path + f"run5/results_run5.csv", index_col = 0, na_values = "np.nan")

# Concatenating the results into one DataFrame. 
results = pd.concat([run1.reset_index(drop = True), run2.reset_index(drop = True),
                     run3.reset_index(drop = True), run4.reset_index(drop = True),
                     run5.reset_index(drop = True)], axis = 0)
    
avg = {}
avg["dem-parity"] = (results["dem-parity"].mean(), results["dem-parity"].std())
avg["dis-impact"] = (results["dis-impact"].mean(), results["dis-impact"].std())
    
for m in metrics:
    if m == "dis":   # averaging over 'regular' values
        m_mean = np.nanmean(results[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]], axis = 1)
        m_std = np.nanstd(results[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]], axis = 1)
    else:   # averaging over absolute values
        m_mean = np.nanmean(abs(results[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]), axis = 1) 
        m_std = np.nanstd(abs(results[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]), axis = 1)
    avg[m] = (m_mean, m_std)
    
max_auroc_idx = results[["rf-auroc", "mlp-auroc", "lda-auroc"]].mean(axis = 1).argmax()
print(f"Synthetic dataset with highest AUROC: sample {max_auroc_idx + 1}")
    
bank_wgangp_set = pd.DataFrame(results.iloc[max_auroc_idx].copy()).T
    
best_avg = {}
    
for m in metrics:
    if m == "dis":   # averaging over 'regular' values
        m_mean = np.nanmean(bank_wgangp_set[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]])
        m_std = np.nanstd(bank_wgangp_set[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]])
    else:   # averaging over absolute values
        m_mean = np.nanmean(abs(bank_wgangp_set[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]])) 
        m_std = np.nanstd(abs(bank_wgangp_set[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]))
    best_avg[m] = (m_mean, m_std)

bank_wgangp_results = pd.concat([bank_wgangp_set[["dem-parity", "dis-impact"]].reset_index(drop = True),
                                 pd.DataFrame(best_avg).reset_index(drop = True)], 
                                 axis = 1).rename(index = {0: "mean", 1: "std"})

In [ ]:
bank_wgangp_results

## Credit Card Default Dataset (Real)

In [ ]:
credit_results_real = pd.read_csv("./train-test-datasets/credit-card-default/results_real.csv", index_col = 0)

results_real_avg = {}
for m in metrics:
    if m == "dis":   # averaging over 'regular' values
        m_mean = credit_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]].mean().mean()
        m_std = credit_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]].mean().std(ddof = 0)
    else:   # averaging over absolute values
        m_mean = abs(credit_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]).mean().mean()
        m_std = abs(credit_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]).mean().std(ddof = 0)
    results_real_avg[m] = (m_mean, m_std)

credit_results = pd.concat([credit_results_real[["dem-parity", "dis-impact"]].reset_index(drop = True),
                            pd.DataFrame(results_real_avg).reset_index(drop = True)], 
                            axis = 1).rename(index = {0: "mean", 1: "std"})

In [ ]:
credit_results

## Synthetic Credit Card Default Dataset (PATE-GAN)

In [ ]:
# Assigning file path for importing the datasets. 
file_path = "synthetic-datasets_PATE-GAN/credit-card-default/"
credit_eps_bests_PT, credit_results_best_PT, credit_results_best_PT_std, results_eps_avg, results_eps_avg_std = results_analysis(file_path,
                                                                                                                                 epsilons = epsi)

In [ ]:
credit_results_best_PT

## Synthetic Credit Card Default Dataset (DP-GAN)

In [ ]:
# Assigning file path for importing the datasets. 
file_path = "synthetic-datasets_DP-GAN_B=512/credit-card-default/"
credit_eps_bests_DP, credit_results_best_DP, credit_results_best_DP_std, results_eps_avg, results_eps_avg_std = results_analysis(file_path,
                                                                                                                                 epsilons = epsi) 

In [ ]:
credit_results_best_DP

## Synthetic Credit Card Default Dataset (WGAN-GP)

In [ ]:
# Assigning file path for importing the datasets. 
file_path = "synthetic-datasets_WGAN-GP/credit-card-default/"

# Importing synthetic datasets' results as pandas DataFrames.
run1 = pd.read_csv(file_path + f"run1/results_run1.csv", index_col = 0, na_values = "np.nan")
run2 = pd.read_csv(file_path + f"run2/results_run2.csv", index_col = 0, na_values = "np.nan")
run3 = pd.read_csv(file_path + f"run3/results_run3.csv", index_col = 0, na_values = "np.nan")
run4 = pd.read_csv(file_path + f"run4/results_run4.csv", index_col = 0, na_values = "np.nan")
run5 = pd.read_csv(file_path + f"run5/results_run5.csv", index_col = 0, na_values = "np.nan")

# Concatenating the results into one DataFrame. 
results = pd.concat([run1.reset_index(drop = True), run2.reset_index(drop = True),
                     run3.reset_index(drop = True), run4.reset_index(drop = True),
                     run5.reset_index(drop = True)], axis = 0)
    
avg = {}
avg["dem-parity"] = (results["dem-parity"].mean(), results["dem-parity"].std())
avg["dis-impact"] = (results["dis-impact"].mean(), results["dis-impact"].std())
    
for m in metrics:
    if m == "dis":   # averaging over 'regular' values
        m_mean = np.nanmean(results[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]], axis = 1)
        m_std = np.nanstd(results[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]], axis = 1)
    else:   # averaging over absolute values
        m_mean = np.nanmean(abs(results[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]), axis = 1) 
        m_std = np.nanstd(abs(results[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]), axis = 1)
    avg[m] = (m_mean, m_std)
    
max_auroc_idx = results[["rf-auroc", "mlp-auroc", "lda-auroc"]].mean(axis = 1).argmax()
print(f"Synthetic dataset with highest AUROC: sample {max_auroc_idx + 1}")
    
credit_wgangp_set = pd.DataFrame(results.iloc[max_auroc_idx].copy()).T
    
best_avg = {}
    
for m in metrics:
    if m == "dis":   # averaging over 'regular' values
        m_mean = np.nanmean(credit_wgangp_set[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]])
        m_std = np.nanstd(credit_wgangp_set[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]])
    else:   # averaging over absolute values
        m_mean = np.nanmean(abs(credit_wgangp_set[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]])) 
        m_std = np.nanstd(abs(credit_wgangp_set[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]))
    best_avg[m] = (m_mean, m_std)

credit_wgangp_results = pd.concat([credit_wgangp_set[["dem-parity", "dis-impact"]].reset_index(drop = True),
                                   pd.DataFrame(best_avg).reset_index(drop = True)], 
                                   axis = 1).rename(index = {0: "mean", 1: "std"})

In [ ]:
credit_wgangp_results

## Average of Results

In [ ]:
# Average of datasets' demographic parity. 
avg_dem_real = [adult_results["dem-parity"][0], bank_results["dem-parity"][0], credit_results["dem-parity"][0]]
avg_dem_wgan = [adult_wgangp_results["dem-parity"][0], bank_wgangp_results["dem-parity"][0], credit_wgangp_results["dem-parity"][0]]
avg_dem_pate = [[adult_results_best_PT["dem-parity"][e], bank_results_best_PT["dem-parity"][e],
                 credit_results_best_PT["dem-parity"][e]] for e in range(len(epsi))]
avg_dem_dp = [[adult_results_best_DP["dem-parity"][e], bank_results_best_DP["dem-parity"][e], 
               credit_results_best_DP["dem-parity"][e]] for e in range(len(epsi))]

# Average of datasets' disparate impact. 
avg_dis_real = [adult_results["dis-impact"][0], bank_results["dis-impact"][0], credit_results["dis-impact"][0]]
avg_dis_wgan = [adult_wgangp_results["dis-impact"][0], bank_wgangp_results["dis-impact"][0], credit_wgangp_results["dis-impact"][0]]
avg_dis_pate = [[adult_results_best_PT["dis-impact"][e], bank_results_best_PT["dis-impact"][e],
                 credit_results_best_PT["dis-impact"][e]] for e in range(len(epsi))]
avg_dis_dp = [[adult_results_best_DP["dis-impact"][e], bank_results_best_DP["dis-impact"][e],
               credit_results_best_DP["dis-impact"][e]] for e in range(len(epsi))]

avg_dem_results = pd.DataFrame({"mean": [np.mean(avg_dem_real), np.mean(avg_dem_wgan)] + list(np.mean(avg_dem_pate, axis = 1)),
                                        "std": [np.std(avg_dem_real), np.std(avg_dem_wgan)] + list(np.std(avg_dem_pate, axis = 1))},
                                       index = ["Real", "WGAN-GP", "epsi_1", "epsi_2", "epsi_5", "epsi_8"]).T

avg_dis_results = pd.DataFrame({"mean": [np.mean(avg_dis_real), np.mean(avg_dis_wgan)] + list(np.mean(avg_dis_pate, axis = 1)), 
                                        "std": [np.std(avg_dis_real), np.std(avg_dis_wgan)] + list(np.std(avg_dis_pate, axis = 1))},
                                       index = ["Real", "WGAN-GP", "epsi_1", "epsi_2", "epsi_5", "epsi_8"]).T

avg_dem_results, avg_dis_results, avg_dis_real, avg_dis_wgan, np.mean(avg_dis_dp, axis = 1)

In [ ]:
# Average of the datasets' utility and fairness metrics (w.r.t. the trained classifiers).
metrics = ["acc", "prec", "recall", "auroc", "dem", "dis", "eqopp"]

avg_real_results = {}
for m in metrics:
    if m != "dis":   # averaging over absolute values
        avg = [abs(adult_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]), abs(bank_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]),
               abs(credit_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]])]
    else:   # averaging over 'regular' values
        avg = [adult_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]], bank_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]],
               credit_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]]
    avg_real_results[m] = (np.mean(avg), np.std(avg))
avg_real_results = pd.DataFrame(avg_real_results).rename(index = {0: "mean", 1: "std"})

avg_met_results_PT, avg_met_results_PT_std = metrics_average(adult_samp = pd.concat([pd.DataFrame(adult_eps_bests_PT[i]).T for i in range(4)]), 
                                                             bank_samp = pd.concat([pd.DataFrame(bank_eps_bests_PT[i]).T for i in range(4)]), 
                                                             credit_samp = pd.concat([pd.DataFrame(credit_eps_bests_PT[i]).T for i in range(4)]))

# Making sure that a "NaN" value is adequately handled. 
bank_eps_bests_DP[1] = bank_eps_bests_DP[1].replace("NaN", np.nan)
avg_met_results_DP, avg_met_results_DP_std = metrics_average(adult_samp = pd.concat([pd.DataFrame(adult_eps_bests_DP[i]).T for i in range(4)]), 
                                                             bank_samp = pd.concat([pd.DataFrame(bank_eps_bests_DP[i]).T for i in range(4)]), 
                                                             credit_samp = pd.concat([pd.DataFrame(credit_eps_bests_DP[i]).T for i in range(4)]))

avg_wgangp_results = {}
for m in metrics:
    if m != "dis":   # averaging over absolute values
        avg = [abs(adult_wgangp_set[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]), abs(bank_wgangp_set[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]),
               abs(credit_wgangp_set[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]])]
    else:   # averaging over 'regular' values
        avg = [adult_wgangp_set[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]], bank_wgangp_set[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]],
               credit_wgangp_set[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]]
    avg_wgangp_results[m] = (np.mean(avg), np.std(avg))
avg_wgangp_results = pd.DataFrame(avg_wgangp_results).rename(index = {0: "mean", 1: "std"})

## Plots of Results

In [ ]:
# Plotting fairness metrics of the synthetic datasets from PATE-GAN and DP-GAN.
positions = [0, 1, 2, 3, 4, 5]
width = 0.25

fairness_clf = ["dem-parity", "dis-impact"]
fairness_labels = ["Demographic Parity", "Disparate Impact"]

for f in fairness_clf:
    # Create a figure for the aggregated plot.
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.0))  # 1 x 4 grid
    
    for c in range(0, 4):
        if c == 0:
            dataset = "Adult"
            results_np = pd.concat([adult_results[f], adult_wgangp_results[f]], axis = 1).set_axis(["Real", "WGAN-GP"], axis = 1)
            
            results_PT = pd.concat([adult_results_best_PT[f], adult_results_best_PT_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_npPT = pd.concat([results_np, results_PT], axis = 1)

            results_DP = pd.concat([adult_results_best_DP[f], adult_results_best_DP_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_npDP = pd.concat([results_np, results_DP], axis = 1)
        elif c == 1:
            dataset = "Bank"
            results_np = pd.concat([bank_results[f], bank_wgangp_results[f]], axis = 1).set_axis(["Real", "WGAN-GP"], axis = 1)
            
            results_PT = pd.concat([bank_results_best_PT[f], bank_results_best_PT_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_npPT = pd.concat([results_np, results_PT], axis = 1)

            results_DP = pd.concat([bank_results_best_DP[f], bank_results_best_DP_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_npDP = pd.concat([results_np, results_DP], axis = 1)
        elif c == 2:
            dataset = "Credit"
            results_np = pd.concat([credit_results[f], credit_wgangp_results[f]], axis = 1).set_axis(["Real", "WGAN-GP"], axis = 1)
            
            results_PT = pd.concat([credit_results_best_PT[f], credit_results_best_PT_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_npPT = pd.concat([results_np, results_PT], axis = 1)

            results_DP = pd.concat([credit_results_best_DP[f], credit_results_best_DP_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_npDP = pd.concat([results_np, results_DP], axis = 1)
        else:
            dataset = "Average"
            if f == "dem-parity":
                results_npPT = pd.DataFrame({"mean": [np.mean(avg_dem_real), np.mean(avg_dem_wgan)] + list(np.mean(avg_dem_pate, axis = 1)),
                                            "std": [np.std(avg_dem_real), np.std(avg_dem_wgan)] + list(np.std(avg_dem_pate, axis = 1))},
                                            index = ["Real", "WGAN-GP", "epsi_1", "epsi_2", "epsi_5", "epsi_8"]).T

                results_npDP = pd.DataFrame({"mean": [np.mean(avg_dem_real), np.mean(avg_dem_wgan)] + list(np.mean(avg_dem_dp, axis = 1)),
                                            "std": [np.std(avg_dem_real), np.std(avg_dem_wgan)] + list(np.std(avg_dem_dp, axis = 1))},
                                            index = ["Real", "WGAN-GP", "epsi_1", "epsi_2", "epsi_5", "epsi_8"]).T
            else: 
                results_npPT = pd.DataFrame({"mean": [np.mean(avg_dis_real), np.mean(avg_dis_wgan)] + list(np.mean(avg_dis_pate, axis = 1)), 
                                            "std": [np.std(avg_dis_real), np.std(avg_dis_wgan)] + list(np.std(avg_dis_pate, axis = 1))},
                                            index = ["Real", "WGAN-GP", "epsi_1", "epsi_2", "epsi_5", "epsi_8"]).T

                results_npDP = pd.DataFrame({"mean": [np.mean(avg_dis_real), np.mean(avg_dis_wgan)] + list(np.mean(avg_dis_dp, axis = 1)), 
                                            "std": [np.std(avg_dis_real), np.std(avg_dis_wgan)] + list(np.std(avg_dis_dp, axis = 1))},
                                            index = ["Real", "WGAN-GP", "epsi_1", "epsi_2", "epsi_5", "epsi_8"]).T
        
    
        col = fairness_clf.index(f)
        ax = axes[c]

        # Plotting a dashed line for the demographic parity and disparate impact parameters.
        ax.axhline(0, 0, 5, ls = "--", c = "black")

        if f == "dis-impact":
            # Plotting another dashed line for the disparate impact parameter.
            ax.axhline(0.45, 0, 5, ls = "--", c = "black")
            # Plotting three rectangular patches for the disparate impact parameter.
            rect1 = mpatches.Rectangle(xy = (-0.5, 0), width = 6.5, height = 0.45, color='green', alpha = 0.1, ec='green')
            ax.add_patch(rect1)
            rect2 = mpatches.Rectangle(xy = (-0.5, 0), width = 6.5, height = -1.5, color='red', alpha = 0.1, ec='red')
            ax.add_patch(rect2)
            rect3 = mpatches.Rectangle(xy = (-0.5, 0.45), width = 6.5, height = 1.5, color='red', alpha = 0.1, ec='red')
            ax.add_patch(rect3)
    
        # Create barplots for the current metric: Adult, Bank, Credit & Average.
        for p in positions: 
            if c != 3:
                if p == 0:
                    rects = ax.bar(p, results_npPT.iloc[0][p], 0.25, color = "red")   # original
                    ax.bar_label(rects, labels = [" "], padding = 3)
                elif p == 1:
                    rects = ax.bar(p, results_npPT.iloc[0][p], 0.25, color = "blue")   # WGAN-GP
                    ax.bar_label(rects, labels = [" "], padding = 3)
                else:
                    rects = ax.bar([p - 0.5*width, p + 0.5*width], [results_npPT.iloc[0][p], results_npDP.iloc[0][p]], width, 
                                   color = ["olive", "purple"])   # PATE-GAN & DP-GAN
                    ax.bar_label(rects, labels = [" ", " "], padding = 3)
            else:
                if p == 0:
                    rects = ax.bar(p, results_npPT.iloc[0][p], 0.25, yerr = results_npPT.iloc[1][p], color = "red", capsize = 2)   # original
                    ax.bar_label(rects, labels = [" "], padding = 3)
                elif p == 1:
                    rects = ax.bar(p, results_npPT.iloc[0][p], 0.25, yerr = results_npPT.iloc[1][p], color = "blue", capsize = 2)   # WGAN-GP
                    ax.bar_label(rects, labels = [" "], padding = 3)
                else: 
                    rects = ax.bar([p - 0.5*width, p + 0.5*width], [results_npPT.iloc[0][p], results_npDP.iloc[0][p]], width,   # PATE-GAN & DP-GAN
                                   color = ["olive", "purple"], yerr = [results_npPT.iloc[1][p], results_npDP.iloc[1][p]], capsize = 2)
                    ax.bar_label(rects, labels = [" ", " "], padding = 3)
            
        # Customizing the plot.
        if c == 0:
            if f == "dis-impact":
                ax.set_ylabel(fairness_labels[col])
            else:
                ax.set_ylabel(f"$|${fairness_labels[col]}$|$")
        ax.set_aspect('auto')
        ax.set_xlim([-0.5, 5.5])
        ax.set_title(f"{dataset}")
        ax.xaxis.set_major_locator(ticker.FixedLocator(positions))
        ax.xaxis.set_major_formatter(ticker.FixedFormatter(["Org.", "WGAN", r"$\epsilon = 1$", r"$\epsilon = 2$", r"$\epsilon = 5$", 
                                                            r"$\epsilon = 8$"]))
        ax.set_xlabel("(GP)", x = 0.26)
        if f == "dem-parity": 
            ax.set_ylim(-0.05, 0.86)
        else:
            ax.set_ylim(-1.05, 1.05)
        ax.grid(True, linestyle = '--', alpha = 0.6)

        if f == "dem-parity":
            olive_patch = mpatches.Patch(color = "olive", label = "PATE-GAN")
            purple_patch = mpatches.Patch(color = "purple", label = "DP-GAN")
            plt.figlegend(handles = [olive_patch, purple_patch], ncol = 2, loc = "upper center", bbox_to_anchor = (0.5, 1.05))
        
        plt.tight_layout(rect = [0, 0, 1, 0.95])
    plt.show()

In [ ]:
# Plotting utility metrics of classifiers trained on the synthetic datasets from PATE-GAN and DP-GAN.
positions = [0, 1, 2, 3, 4, 5]
width = 0.25

utility_clf = ["acc", "auroc"]
utility_labels = ["Accuracy (\%)", "AUROC"]

for u in utility_clf:
    # Creating a figure for the aggregated plot.
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.0))  # 1 x 4 grid
    for c in range(0, 4):
        if c == 0:
            dataset = "Adult"
            results_np = pd.concat([adult_results[u], adult_wgangp_results[u]], axis = 1).set_axis(["Real", "WGAN-GP"], axis = 1)
            
            results_PT = pd.concat([adult_results_best_PT[u], adult_results_best_PT_std[u]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_npPT = pd.concat([results_np, results_PT], axis = 1)

            results_DP = pd.concat([adult_results_best_DP[u], adult_results_best_DP_std[u]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_npDP = pd.concat([results_np, results_DP], axis = 1)
        elif c == 1:
            dataset = "Bank"
            results_np = pd.concat([bank_results[u], bank_wgangp_results[u]], axis = 1).set_axis(["Real", "WGAN-GP"], axis = 1)
            
            results_PT = pd.concat([bank_results_best_PT[u], bank_results_best_PT_std[u]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_npPT = pd.concat([results_np, results_PT], axis = 1)

            results_DP = pd.concat([bank_results_best_DP[u], bank_results_best_DP_std[u]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_npDP = pd.concat([results_np, results_DP], axis = 1)
        elif c == 2:
            dataset = "Credit"
            results_np = pd.concat([credit_results[u], credit_wgangp_results[u]], axis = 1).set_axis(["Real", "WGAN-GP"], axis = 1)
            
            results_PT = pd.concat([credit_results_best_PT[u], credit_results_best_PT_std[u]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_npPT = pd.concat([results_np, results_PT], axis = 1)

            results_DP = pd.concat([credit_results_best_DP[u], credit_results_best_DP_std[u]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_npDP = pd.concat([results_np, results_DP], axis = 1)
        else:
            dataset = "Average"
            results_np = pd.concat([avg_real_results[u], avg_wgangp_results[u]], axis = 1).set_axis(["Real", "WGAN-GP"], axis = 1)
            
            results_PT = pd.concat([avg_met_results_PT[u], avg_met_results_PT_std[u]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_npPT = pd.concat([results_np, results_PT], axis = 1)

            results_DP = pd.concat([avg_met_results_DP[u], avg_met_results_DP_std[u]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_npDP = pd.concat([results_np, results_DP], axis = 1)
    
        idx = utility_clf.index(u)
        ax = axes[c]
    
        # Create barplots for the current metric: Adult, Bank, Credit & Average.
        for p in positions:
            if p == 0:
                rects = ax.bar(p, results_npPT.iloc[0][p], 0.25, yerr = results_npPT.iloc[1][p], color = "red", capsize = 2)   # original
                ax.bar_label(rects, labels = [" "], padding = 3)
            elif p == 1:
                rects = ax.bar(p, results_npPT.iloc[0][p], 0.25, yerr = results_npPT.iloc[1][p], color = "blue", capsize = 2)   # WGAN-GP
                ax.bar_label(rects, labels = [" "], padding = 3)
            else: 
                rects = ax.bar([p - 0.5*width, p + 0.5*width], [results_npPT.iloc[0][p], results_npDP.iloc[0][p]], width, 
                                color = ["olive", "purple"], yerr = [results_npPT.iloc[1][p], results_npDP.iloc[1][p]],
                               capsize = 2)   # PATE-GAN & DP-GAN
                ax.bar_label(rects, labels = [" ", " "], padding = 3)
            
        # Customizing the plot.
        if c == 0:
            ax.set_ylabel(utility_labels[idx])
        ax.set_xlim([-0.5, 5.5])
        if u == "auroc":
            ax.set_ylim([0, 0.83])
        else:
            ax.set_ylim([0, 96])
        ax.set_title(f"{dataset}")
        ax.xaxis.set_major_locator(ticker.FixedLocator(positions))
        ax.xaxis.set_major_formatter(ticker.FixedFormatter(["Org.", "WGAN", r"$\epsilon = 1$", r"$\epsilon = 2$", r"$\epsilon = 5$", 
                                                            r"$\epsilon = 8$"]))
        ax.set_xlabel("(GP)", x = 0.26)
        ax.grid(True, linestyle='--', alpha=0.6)

        if u == "acc":
            olive_patch = mpatches.Patch(color = "olive", label = "PATE-GAN")
            purple_patch = mpatches.Patch(color = "purple", label = "DP-GAN")
            plt.figlegend(handles = [olive_patch, purple_patch], ncol = 2, loc = "upper center", bbox_to_anchor = (0.5, 1.05))
        
        plt.tight_layout(rect = [0, 0, 1, 0.95])
    plt.show()

In [ ]:
# Plotting fairness metrics of classifiers trained on the synthetic datasets from PATE-GAN and DP-GAN.
positions = [0, 1, 2, 3, 4, 5]
width = 0.25

fairness_clf = ["dem", "dis", "eqopp"]
fairness_labels = ["Demographic Parity", "Disparate Impact", "Equal Opportunity"]

for f in fairness_clf:
    # Creating a figure for the aggregated plot.
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.0))  # 1x4 grid
    for c in range(0, 4):
        if c == 0:
            dataset = "Adult"
            results_np = pd.concat([adult_results[f], adult_wgangp_results[f]], axis = 1).set_axis(["Real", "WGAN-GP"], axis = 1)
            
            results_PT = pd.concat([adult_results_best_PT[f], adult_results_best_PT_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_npPT = pd.concat([results_np, results_PT], axis = 1)

            results_DP = pd.concat([adult_results_best_DP[f], adult_results_best_DP_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_npDP = pd.concat([results_np, results_DP], axis = 1)
        elif c == 1:
            dataset = "Bank"
            results_np = pd.concat([bank_results[f], bank_wgangp_results[f]], axis = 1).set_axis(["Real", "WGAN-GP"], axis = 1)
            
            results_PT = pd.concat([bank_results_best_PT[f], bank_results_best_PT_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_npPT = pd.concat([results_np, results_PT], axis = 1)

            results_DP = pd.concat([bank_results_best_DP[f], bank_results_best_DP_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_npDP = pd.concat([results_np, results_DP], axis = 1)
        elif c == 2:
            dataset = "Credit"
            results_np = pd.concat([credit_results[f], credit_wgangp_results[f]], axis = 1).set_axis(["Real", "WGAN-GP"], axis = 1)
            
            results_PT = pd.concat([credit_results_best_PT[f], credit_results_best_PT_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_npPT = pd.concat([results_np, results_PT], axis = 1)

            results_DP = pd.concat([credit_results_best_DP[f], credit_results_best_DP_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_npDP = pd.concat([results_np, results_DP], axis = 1)
        else:
            dataset = "Average"
            results_np = pd.concat([avg_real_results[f], avg_wgangp_results[f]], axis = 1).set_axis(["Real", "WGAN-GP"], axis = 1)
            
            results_PT = pd.concat([avg_met_results_PT[f], avg_met_results_PT_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_npPT = pd.concat([results_np, results_PT], axis = 1)

            results_DP = pd.concat([avg_met_results_DP[f], avg_met_results_DP_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_npDP = pd.concat([results_np, results_DP], axis = 1)
    
        col = fairness_clf.index(f)
        ax = axes[c]

        # Plotting a dashed line for the demographic parity, disparate impact and equal opportunity parameters.
        ax.axhline(0, 0, 5, ls = "--", c = "black")

        if f == "dis":
            # Plotting another dashed line for the disparate impact parameter.
            ax.axhline(0.45, 0, 5, ls = "--", c = "black")
            # Plotting three rectangular patches for the disparate impact parameter.
            rect1 = mpatches.Rectangle(xy = (-0.5, 0), width = 6.5, height = 0.45, color='green', alpha = 0.1, ec='green')
            ax.add_patch(rect1)
            rect2 = mpatches.Rectangle(xy = (-0.5, 0), width = 6.5, height = -10.5, color='red', alpha = 0.1, ec='red')
            ax.add_patch(rect2)
            rect3 = mpatches.Rectangle(xy = (-0.5, 0.45), width = 6.5, height = 10.5, color='red', alpha = 0.1, ec='red')
            ax.add_patch(rect3)
    
        # Create barplots for the current metric: Adult, Bank, Credit & Average.
        for p in positions: 
            if p == 0:
                rects = ax.bar(p, results_npPT.iloc[0][p], width, yerr = results_npPT.iloc[1][p], color = "red", capsize = 2)   # original
                ax.bar_label(rects, labels = [" "], padding = 3)
            elif p == 1:
                rects = ax.bar(p, results_npPT.iloc[0][p], width, yerr = results_npPT.iloc[1][p], color = "blue", capsize = 2)   # WGAN-GP
                ax.bar_label(rects, labels = [" "], padding = 3)
            else: 
                rects = ax.bar([p - 0.5*width, p + 0.5*width], [results_npPT.iloc[0][p], results_npDP.iloc[0][p]], width,   # PATE-GAN & DP-GAN
                               color = ["olive", "purple"], yerr = [results_npPT.iloc[1][p], results_npDP.iloc[1][p]], capsize = 2)   
                ax.bar_label(rects, labels = [" ", " "], padding = 3)
            
        # Customizing the plot.
        if c == 0:
            if f == "dis":
                ax.set_ylabel(fairness_labels[col])
            else:
                ax.set_ylabel(f"$|${fairness_labels[col]}$|$")
        ax.set_aspect('auto')
        ax.set_xlim([-0.5, 5.5])
        ax.set_title(f"{dataset}")
        ax.xaxis.set_major_locator(ticker.FixedLocator(positions))
        ax.xaxis.set_major_formatter(ticker.FixedFormatter(["Org.", "WGAN", r"$\epsilon = 1$", r"$\epsilon = 2$", r"$\epsilon = 5$", 
                                                            r"$\epsilon = 8$"]))
        ax.set_xlabel("(GP)", x = 0.26)
        if f == "dis": 
            ax.set_ylim(-1.05, 1.05)
        elif f == "eqopp":
            ax.set_ylim(-0.03, 0.56)      
        else:
            ax.set_ylim(-0.03, 0.6)

        ax.grid(True, linestyle = '--', alpha = 0.6)

        if f == "dem":
            olive_patch = mpatches.Patch(color = "olive", label = "PATE-GAN")
            purple_patch = mpatches.Patch(color = "purple", label = "DP-GAN")
            plt.figlegend(handles = [olive_patch, purple_patch], ncol = 2, loc = "upper center", bbox_to_anchor = (0.5, 1.05))
        
        plt.tight_layout(rect = [0, 0, 1, 0.95])
    plt.show()